In [10]:
import requests
import pandas as pd

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)  # show all columns
pd.set_option('display.width', None)        # don't wrap output

In [11]:
# this is a method for querying the SDA with a SQL method and returning the result as a dataframe
def query_sda(sql):
    url = "https://SDMDataAccess.sc.egov.usda.gov/Tabular/post.rest"
    payload = {"query": sql, "format": "json+columnname"}
    
    response = requests.post(url, data=payload)
    response.raise_for_status()
    
    if not response.text:
        raise ValueError("Empty response from SDA — check your SQL for errors")
    
    data = response.json()
    
    # The response wraps results in a 'Table' key (or 'Table1', 'Table2'... for multiple result sets)
    # First row is column names when using json+columnname
    table = data.get("Table", [])
    if not table:
        return pd.DataFrame()
    
    columns = table[0]
    rows = table[1:]
    return pd.DataFrame(rows, columns=columns)


# Sanity check — just fetch 5 map units
sql = "SELECT TOP 5 mukey, muname FROM mapunit"
df = query_sda(sql)
print(df)

   mukey                                                   muname
0  49456       Sarkar-McGilvery complex, 75 to 120 percent slopes
1  49457    Kupreanof-Tolstoi association, 5 to 35 percent slopes
2  49458   Kupreanof-Tolstoi association, 35 to 75 percent slopes
3  49459  Kupreanof-Tolstoi association, 75 to 100 percent slopes
4  49460        Liesnoi-Blaquirre complex, 5 to 35 percent slopes


In [12]:
def get_location_data_from_point(lon, lat):
    sql = f"""
    SELECT mu.mukey, mu.muname, l.areasymbol
    FROM mapunit mu
    JOIN legend l ON mu.lkey = l.lkey
    WHERE mu.mukey IN (
        SELECT * FROM SDA_Get_Mukey_from_intersection_with_WktWgs84(
            'POINT({lon} {lat})'
        )
    )
    """
    return query_sda(sql)

# Des Moines, Iowa
result = get_location_data_from_point(-93.6250, 41.5868)
print(result)

    mukey      muname areasymbol
0  412895  Urban land      IA153


In [13]:
sql = """SELECT TOP 5 * FROM cocropyld"""
dftest = query_sda(sql)
print(dftest.columns.tolist())  # see real column names
print(dftest.head())
print(dftest.columns)

['cropname', 'yldunits', 'nonirryield_l', 'nonirryield_r', 'nonirryield_h', 'irryield_l', 'irryield_r', 'irryield_h', 'cropprodindex', 'vasoiprdgrp', 'cokey', 'cocropyldkey']
        cropname yldunits nonirryield_l nonirryield_r nonirryield_h  \
0   Buffel grass      AUM          None             5          None   
1       Avocados    Boxes          None           535          None   
2     Pigeonpeas      Cwt          None            10          None   
3  Green peppers     Tons          None           NaN          None   
4       Tomatoes     Tons          None           NaN          None   

  irryield_l irryield_r irryield_h cropprodindex vasoiprdgrp     cokey  \
0       None        NaN       None          None        None  27626831   
1       None        NaN       None          None        None  27626831   
2       None        NaN       None          None        None  27626831   
3       None          8       None          None        None  27626831   
4       None         10     

In [14]:
# get all relevant tables associated to the map unit containing the specified point
# Currently gets
#   - crop data from cocropyld
#   - horizons data from chorizons
#   - restrictions data from corestrictions
def get_tables_from_point(lon, lat):
    mukey = get_location_data_from_point(lon, lat).iloc[0]["mukey"]

    # crop data 
    crop_sql = f"""
        SELECT co.compname, co.comppct_r, cc.*
        FROM component co
        JOIN cocropyld cc ON co.cokey = cc.cokey
        WHERE co.mukey = '{mukey}'"""

    # horizons
    horizons_sql = f"""
        SELECT co.compname, co.comppct_r, ch.*
        FROM component co
        JOIN chorizon ch ON co.cokey = ch.cokey
        WHERE co.mukey = '{mukey}'"""

    # Restrictions data
    restrictions_sql = f"""
        SELECT co.compname, co.comppct_r, cr.*
        FROM component co
        JOIN corestrictions cr ON co.cokey = cr.cokey
        WHERE co.mukey = '{mukey}'"""
    return query_sda(crop_sql), query_sda(horizons_sql), query_sda(restrictions_sql)

lon, lat = -93.6250, 41.5868
print(get_location_data_from_point(lon, lat).iloc[0]["mukey"])
print(get_tables_from_point(lon, lat))

412895
(Empty DataFrame
Columns: []
Index: [], Empty DataFrame
Columns: []
Index: [], Empty DataFrame
Columns: []
Index: [])


In [15]:
import plotly.express as px
import pandas as pd

def plot_point(lat, lon):
    df = pd.DataFrame({"lat": [lat], "lon": [lon]})
    fig = px.scatter_map(df, lat="lat", lon="lon", zoom=13)
    fig.show()

plot_point(41.5868, -93.6250)

In [16]:
import folium

def plot_point(lat, lon, zoom=13):
    m = folium.Map(location=[lat, lon], zoom_start=zoom)
    folium.Marker([lat, lon]).add_to(m)
    return m

m = plot_point(41.5868, -93.6250)
m  # renders inline in Jupyter

In [17]:
import sda_utils as sda

lon, lat = -93.647713, 37.774576
print(sda.get_location_data_from_point(lat, lon))
crop, horizon, restriction = sda.get_tables_from_point(lat, lon)
print(crop)

     mukey                                         muname areasymbol
0  2533890  Cliquot gravelly loam, 8 to 15 percent slopes      MO039
    compname comppct_r                 cropname yldunits nonirryield_l  \
0    Viraton         4                     Corn       Bu          None   
1    Viraton         4              Alfalfa hay     Tons          None   
2    Viraton         4       Caucasian bluestem     Tons          None   
3    Viraton         4      Common bermudagrass     Tons          None   
4    Viraton         4  Orchardgrass-red clover     Tons          None   
5    Viraton         4              Tall fescue     Tons          None   
6    Viraton         4      Warm season grasses     Tons          None   
7   Cedargap         3              Alfalfa hay     Tons          None   
8   Cedargap         3       Caucasian bluestem     Tons          None   
9   Cedargap         3      Common bermudagrass     Tons          None   
10  Cedargap         3  Orchardgrass-red clover 

In [18]:
print(sda.summarize_crop_yields(crop))

                        yldunits  nonirryield_r irryield_r
cropname                                                  
Alfalfa hay                 Tons          13.64          0
Caucasian bluestem          Tons          25.70          0
Common bermudagrass         Tons          25.35          0
Corn                          Bu           0.00          0
Orchardgrass-red clover     Tons          25.05          0
Tall fescue                 Tons          26.60          0
Warm season grasses         Tons          32.40          0
